In [ ]:
"""
=============================================================
INSURANCE POLICY RECOMMENDATION PIPELINE
=============================================================

Procjena situacije:
- 54k klijenata, median=1 polisa, Q3=2 polisa
- Dostupno: demografija (starost, pol, svojina, lokacija)
- Dostupno: podaci o polisama (vrsta, premija, suma, trajanje)
- Dostupno: ind_steta (signal o štetama)

Zaključak:
- Sliding window / sequential modeli OTPADAJU (median=1)
- Apriori sam OTPADA (ignoruje demografiju)
- LightFM Hybrid (Content + Collaborative) je OPTIMALAN:
    → Content-based riješava cold start za klijente sa 1 polisom
    → Collaborative uči iz ponašanja sličnih klijenata
    → WARP loss optimizuje ranking direktno (top-N preporuke)
    → Jedna biblioteka, jedan model, sve u jednom

Instalacija:
    pip install lightfm pandas numpy scikit-learn
=============================================================
"""

import pandas as pd
import numpy as np
from datetime import datetime
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, auc_score
import warnings
warnings.filterwarnings('ignore')


# =============================================================
# KORAK 1: UČITAVANJE
# =============================================================

df = pd.read_csv('polise.csv', low_memory=False)

# Kolone koje koristimo iz tvoje tabele:
# klijent_id, sif_vrsta, premija_ukupno, suma_osiguranja,
# dat_od, dat_do, br_dana, ind_steta, konacna_premija,
# osig_svojina_cli, osig_opstina_cli, osig_mesto_cli,
# datum_rodjenja, pol_id, sif_bracni_status, sif_delatnost


# =============================================================
# KORAK 2: ČIŠĆENJE I FEATURE ENGINEERING
# =============================================================

def izvuci_starost(row):
    """Starost iz datum_rodjenja ili JMBG ako datum nije dostupan."""
    try:
        if pd.notna(row.get('datum_rodjenja')):
            god = pd.to_datetime(row['datum_rodjenja']).year
            return datetime.today().year - god
    except:
        pass
    try:
        jmbg = str(row.get('osig_jmbg_cli', ''))
        if len(jmbg) == 13 and jmbg.isdigit():
            god = int(jmbg[4:7])
            god = 1900 + god if god > 25 else 2000 + god
            return datetime.today().year - god
    except:
        pass
    return np.nan

def starost_grupa(starost):
    if pd.isna(starost):      return 'nepoznato'
    if starost < 25:          return '18-24'
    if starost < 35:          return '25-34'
    if starost < 45:          return '35-44'
    if starost < 55:          return '45-54'
    if starost < 65:          return '55-64'
    return '65+'

def premija_grupa(premija):
    if pd.isna(premija):      return 'nepoznato'
    if premija < 100:         return 'niska'
    if premija < 500:         return 'srednja'
    if premija < 2000:        return 'visoka'
    return 'vrlo_visoka'

# Starost
df['starost'] = df.apply(izvuci_starost, axis=1)
df['starost_grupa'] = df['starost'].apply(starost_grupa)

# Pol
df['pol'] = df['pol_id'].map({1: 'muski', 2: 'zenski'}).fillna('nepoznato')

# Svojina
df['svojina'] = df['osig_svojina_cli'].map({1: 'fizicko', 2: 'pravno'}).fillna('nepoznato')

# Lokacija – opstina kao kategorija
df['opstina'] = df['osig_opstina_cli'].fillna(0).astype(int).astype(str)

# Premija grupa
df['premija_grupa'] = df['premija_ukupno'].apply(premija_grupa)

# Šteta signal
df['ima_stetu'] = df['ind_steta'].fillna('N').apply(lambda x: 'ima_stetu' if x == 'D' else 'bez_stete')

# Bračni status
df['bracni'] = df['sif_bracni_status'].fillna('nepoznato').astype(str)

# Djelatnost (uglavnom za pravna lica)
df['delatnost'] = df['sif_delatnost'].fillna('nepoznato').astype(str)

# Vrsta polise kao string (target item)
df['vrsta_polise'] = df['sif_vrsta'].astype(str)

# Broj polisa po klijentu (signal aktivnosti)
broj_polisa = df.groupby('klijent_id')['vrsta_polise'].count().reset_index()
broj_polisa.columns = ['klijent_id', 'ukupno_polisa']
df = df.merge(broj_polisa, on='klijent_id', how='left')

def aktivnost_grupa(n):
    if n == 1:   return 'novi'
    if n <= 3:   return 'aktivan'
    return 'loyalan'

df['aktivnost'] = df['ukupno_polisa'].apply(aktivnost_grupa)


# =============================================================
# KORAK 3: LIGHTFM DATASET – registracija klijenata i polisa
# =============================================================

dataset = Dataset()

# Sve moguće user features (demografija)
user_feature_names = (
    ['pol_' + x for x in df['pol'].unique()] +
    ['svojina_' + x for x in df['svojina'].unique()] +
    ['starost_' + x for x in df['starost_grupa'].unique()] +
    ['opstina_' + x for x in df['opstina'].unique()] +
    ['premija_' + x for x in df['premija_grupa'].unique()] +
    ['steta_' + x for x in df['ima_stetu'].unique()] +
    ['bracni_' + x for x in df['bracni'].unique()] +
    ['delatnost_' + x for x in df['delatnost'].unique()] +
    ['aktivnost_' + x for x in df['aktivnost'].unique()]
)

dataset.fit(
    users=df['klijent_id'].unique(),
    items=df['vrsta_polise'].unique(),
    user_features=user_feature_names
)

print(f"Klijenata: {df['klijent_id'].nunique()}")
print(f"Vrsta polisa: {df['vrsta_polise'].nunique()}")
print(f"User features: {len(user_feature_names)}")


# =============================================================
# KORAK 4: IZGRADNJA INTERAKCIJA I USER FEATURE MATRICE
# =============================================================

# Interakcije: klijent × polisa (ima/nema)
interactions, weights = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df.iterrows()
])

# User features po klijentu (uzimamo zadnji red za svakog klijenta)
klijent_profil = df.sort_values('dat_od').groupby('klijent_id').last().reset_index()

def napravi_user_features(row):
    """Vraća listu (feature_name, težina) za jednog klijenta."""
    features = [
        ('pol_' + str(row['pol']), 1.0),
        ('svojina_' + str(row['svojina']), 1.0),
        ('starost_' + str(row['starost_grupa']), 1.0),
        ('opstina_' + str(row['opstina']), 1.0),
        ('premija_' + str(row['premija_grupa']), 1.0),
        ('steta_' + str(row['ima_stetu']), 1.0),
        ('bracni_' + str(row['bracni']), 1.0),
        ('delatnost_' + str(row['delatnost']), 1.0),
        ('aktivnost_' + str(row['aktivnost']), 1.0),
    ]
    return features

user_features_list = [
    (row['klijent_id'], napravi_user_features(row))
    for _, row in klijent_profil.iterrows()
]

user_features = dataset.build_user_features(user_features_list)

print(f"Interakcija matrica: {interactions.shape}")
print(f"User features matrica: {user_features.shape}")


# =============================================================
# KORAK 5: TRAIN / TEST SPLIT
# =============================================================
# Strategija: po vremenu – starije polise za trening, novije za test
# Ovo je realnije od random splita za preporuke

df_sorted = df.sort_values('dat_od')
split_datum = df_sorted['dat_od'].quantile(0.8)  # 80% za trening

train_ids = set(df_sorted[df_sorted['dat_od'] <= split_datum]['klijent_id'])
test_ids  = set(df_sorted[df_sorted['dat_od'] >  split_datum]['klijent_id'])

# Za evaluaciju koristimo klijente koji se pojavljuju u oba seta
eval_ids = train_ids & test_ids

train_interactions, _ = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df_sorted[df_sorted['dat_od'] <= split_datum].iterrows()
])

test_interactions, _ = dataset.build_interactions([
    (row['klijent_id'], row['vrsta_polise'])
    for _, row in df_sorted[df_sorted['dat_od'] > split_datum].iterrows()
])


# =============================================================
# KORAK 6: TRENING LIGHTFM MODELA
# =============================================================
# WARP loss = Weighted Approximate-Rank Pairwise
# Optimizuje direktno ranking (top-N), idealno za preporuke
# Bolje od BPR za asimetrične podatke poput osiguranja

model = LightFM(
    loss='warp',
    no_components=64,       # dimenzija embeddings (povećaj za više podataka)
    learning_rate=0.05,
    item_alpha=1e-6,        # L2 regularizacija za items
    user_alpha=1e-6,        # L2 regularizacija za users
    random_state=42
)

print("\nTreniranje modela...")
NUM_EPOCHS = 30

for epoch in range(NUM_EPOCHS):
    model.fit_partial(
        train_interactions,
        user_features=user_features,
        epochs=1,
        num_threads=4,
        verbose=False
    )
    if (epoch + 1) % 10 == 0:
        train_prec = precision_at_k(
            model, train_interactions,
            user_features=user_features, k=3
        ).mean()
        print(f"  Epoch {epoch+1}/{NUM_EPOCHS} | Precision@3 (train): {train_prec:.4f}")

# Finalna evaluacija na test setu
test_prec = precision_at_k(
    model, test_interactions,
    train_interactions=train_interactions,
    user_features=user_features,
    k=3
).mean()

test_auc = auc_score(
    model, test_interactions,
    train_interactions=train_interactions,
    user_features=user_features
).mean()

print(f"\nTest Precision@3: {test_prec:.4f}")
print(f"Test AUC:         {test_auc:.4f}")


# =============================================================
# KORAK 7: PREPORUKE ZA JEDNOG KLIJENTA
# =============================================================

# Mapiranje ID-eva
user_id_map, _, item_id_map, _ = dataset.mapping()
item_id_reverse = {v: k for k, v in item_id_map.items()}

def preporuci_klijentu(klijent_id, top_n=3):
    """
    Vraća top N preporuka za klijenta.
    Automatski isključuje polise koje klijent već ima.
    """
    if klijent_id not in user_id_map:
        return []

    user_idx = user_id_map[klijent_id]
    n_items  = len(item_id_map)

    # Scores za sve polise
    scores = model.predict(
        user_ids=user_idx,
        item_ids=np.arange(n_items),
        user_features=user_features
    )

    # Polise koje klijent već ima
    vec_ima = set(df[df['klijent_id'] == klijent_id]['vrsta_polise'].unique())

    # Rangiraj i filtriraj
    rangirano = sorted(
        [(item_id_reverse[i], scores[i]) for i in range(n_items)
         if item_id_reverse[i] not in vec_ima],
        key=lambda x: x[1],
        reverse=True
    )

    return rangirano[:top_n]


# Primjer
primjer_id = df['klijent_id'].iloc[0]
ima = df[df['klijent_id'] == primjer_id]['vrsta_polise'].unique()
preporuke = preporuci_klijentu(primjer_id)

print(f"\nKlijent: {primjer_id}")
print(f"Ima polise:   {list(ima)}")
print(f"Preporuke:")
for vrsta, score in preporuke:
    print(f"  → Vrsta {vrsta}  (score: {score:.3f})")


# =============================================================
# KORAK 8: BATCH PREPORUKE – SVI KLIJENTI
# =============================================================

print("\nGenerisanje preporuka za sve klijente...")

batch = []
for kid in df['klijent_id'].unique():
    try:
        preporuke = preporuci_klijentu(kid, top_n=3)
        for rang, (vrsta, score) in enumerate(preporuke, 1):
            batch.append({
                'klijent_id': kid,
                'rang':       rang,
                'preporuka_vrsta_polise': vrsta,
                'score':      round(float(score), 4)
            })
    except Exception as e:
        continue

rezultati_df = pd.DataFrame(batch)
rezultati_df.to_csv('preporuke_final.csv', index=False)

print(f"Sačuvano {len(rezultati_df)} preporuka")
print(f"Za {rezultati_df['klijent_id'].nunique()} klijenata")
print(f"\nPrimjer izlaza:")
print(rezultati_df.head(9).to_string(index=False))


# =============================================================
# KORAK 9: OBJAŠNJENJE ZA AGENTA (Apriori pravila)
# =============================================================
# LightFM daje score, ali agent treba objašnjenje.
# Apriori koristimo SAMO za ovo – ne za preporuku.

from mlxtend.frequent_patterns import apriori, association_rules

print("\nGenerisanje objašnjenja (Apriori pravila)...")

# Pivot matrica klijent × polisa
pivot = df.pivot_table(
    index='klijent_id',
    columns='vrsta_polise',
    values='sif_vrsta',
    aggfunc='count'
).fillna(0).gt(0)

frequent_itemsets = apriori(pivot, min_support=0.03, use_colnames=True)

if not frequent_itemsets.empty:
    pravila = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
    pravila = pravila[pravila['confidence'] >= 0.3].sort_values('lift', ascending=False)

    print(f"\nTop 10 pravila:")
    print(pravila[['antecedents','consequents','confidence','lift']].head(10).to_string(index=False))

    pravila.to_csv('apriori_objasnjenja.csv', index=False)


def objasni_preporuku(vrsta_polise, pravila, klijent_polise):
    """Za agenta: zašto preporučujemo ovu polisu."""
    relevantna = pravila[
        pravila['consequents'].apply(lambda x: vrsta_polise in x)
    ]
    for _, pravilo in relevantna.iterrows():
        antecedent = set(pravilo['antecedents'])
        if antecedent.issubset(set(klijent_polise)):
            conf = pravilo['confidence'] * 100
            return (f"Od klijenata koji imaju {list(antecedent)}, "
                    f"{conf:.0f}% ima i polisu vrste {vrsta_polise}.")
    return f"Preporučeno na osnovu profila sličnih klijenata."


# Demo objašnjenja
print(f"\nObjašnjenja za klijenta {primjer_id}:")
for vrsta, score in preporuke:
    objasnjenje = objasni_preporuku(vrsta, pravila, list(ima))
    print(f"  → Vrsta {vrsta}: {objasnjenje}")


print("\n✓ Pipeline završen.")
print("  preporuke_final.csv   – preporuke za sve klijente")
print("  apriori_objasnjenja.csv – pravila za objašnjenje agentima")

Tri ključne odluke
1. LightFM umjesto čistog Collaborative Filtering
Klijent sa 1 polisom nema historiju – ali ima demografiju. LightFM koristi oboje istovremeno u jednom modelu.
2. WARP loss umjesto klasične regresije
Ne predviđa 0/1 – direktno optimizuje ranking. Cilj nije "hoće li uzeti", nego "šta staviti na prvo mjesto od tri preporuke".
3. Apriori samo za objašnjenje
Agent ne može prodati "score 0.73". Može prodati "65% klijenata sa ovim polisama uzme i ovu". LightFM preporučuje, Apriori objašnjava.
Izlaz

preporuke_final.csv – top 3 preporuke po klijentu sa scoreom
apriori_objasnjenja.csv – pravila za agente

Odlično pitanje – hajde da to razložim potpuno iskreno, i tehnički i biznis strana.

Šta je LightFM
LightFM je hibridni model preporuka koji kombinuje dva pristupa u jednom:
Collaborative Filtering (CF)
Ideja: "Klijenti slični tebi su uzeli ovo"

Kako radi:
- Svaki klijent dobija vektor (embedding) u prostoru od 64 dimenzije
- Svaka polisa dobija vektor u istom prostoru
- Ako klijent "ide uz" polisu → njihovi vektori su bliski
- Model uči te vektore iz historije kupovina

Analogija:
Netflix preporuke – ne znaš zašto ti preporučuje film,
ali znaš da su ga gledali ljudi koji su gledali iste filmove kao ti
Content-Based Filtering (CB)
Ideja: "Tvoj profil odgovara ovoj polisi"

Kako radi:
- Demografija klijenta (pol, starost, lokacija...) 
  direktno utiče na embedding
- Dva klijenta sa istim profilom → slični embeddinzi
  čak i ako nemaju zajedničku historiju

Analogija:
Spotify – novi korisnik, nema historije slušanja,
ali zna da si 28 godina, muški → preporučuje na osnovu toga
Kako LightFM kombinuje oba
Tradicionalni CF:
embedding_klijenta = naučen samo iz interakcija
Problem: klijent sa 1 polisom ima slab embedding

LightFM:
embedding_klijenta = suma(embedding za svaki feature)
                   = embedding(muski) 
                   + embedding(25-34)
                   + embedding(fizicko)
                   + embedding(Banja Luka)
                   + embedding(ima_kasko)
                   + ...

Svaki feature doprinosi embeddinzu!
Klijent sa 1 polisom i dalje ima jak embedding
jer demografija kompenzuje nedostatak historije

Šta je WARP loss
Klasičan model uči: "predvidi 1 ili 0"
→ Optimizuje tačnost predikcije

WARP uči: "stavi tačnu preporuku što više u rangu"
→ Optimizuje direktno Precision@K (top 3 preporuke)

Razlika u praksi:
Klasičan: možda dobro predviđa ali na 5. mjestu lista
WARP:     siguran da je prava preporuka u top 3

Za osiguranje agent gleda top 3 → WARP je prirodan izbor

Šta model konkretno radi korak po korak
1. Svaki klijent → vektor od 64 broja
   [0.3, -0.7, 1.2, 0.1, ... ] ← to je "profil" klijenta u prostoru

2. Svaka polisa → vektor od 64 broja
   [0.8, -0.2, 0.9, 0.4, ... ] ← to je "profil" polise

3. Score = dot product (množenje vektora)
   Veći score → veća podudarnost klijent-polisa

4. Top 3 polise sa najvećim scoreom koje klijent nema → preporuka

Kakve rezultate očekujem
Iskreno, sa tvojim podacima:
Precision@3 (glavna metrika)
Što znači: od 3 preporučene polise, koliko ih je klijent stvarno uzeo

Realna očekivanja za tvoje podatke:
Loše:     < 0.10  (1 od 10 preporuka tačna)
Solidno:    0.15  (1-2 od 10 tačne)
Dobro:      0.25  (1 od 4 tačna)
Odlično:  > 0.35  (1 od 3 tačna)

Moja procjena za tvoj slučaj: 0.15 – 0.25
Zašto ne više
Median = 1 polisa po klijentu → malo podataka za učenje
Osiguranje nije Amazon → klijent uzme 2-3 polise u životu,
                          ne 50 proizvoda mjesečno
Signal je slab po prirodi problema
AUC score
Mjeri: da li model rangira tačne polise više od netačnih
Očekujem: 0.70 – 0.85
Prihvatljivo za produkciju: > 0.70

Da li možeš prodati ovo osiguravajućem društvu
Iskrean odgovor je da, ali uz pravu priču. Evo kako:
Šta NE prodaješ
❌ "Model će predvidjeti šta će klijent kupiti"
❌ "Precision je 90%"
❌ "Ovo je AI koji zna sve o klijentu"
Šta PRODAJEŠ
✅ "Bez modela agent nema pojma kome šta da ponudi
    → prodaje nasumično ili po instinktu"

✅ "Sa modelom agent ima prioritizovanu listu
    → fokusira se na klijente sa visokim scoreom"

✅ "Čak i 15% precision znači:
    Ako imate 50.000 klijenata i 3 preporuke po klijentu
    = 150.000 preporuka
    × 15% tačnost
    = 22.500 potencijalnih cross-sell polisa
    × prosječna premija npr. 300 KM
    = 6.750.000 KM potencijalnog prihoda"
Najjači argument
Baseline (bez modela):
Agent pozove random klijenta i random ponudi polisu
Hit rate: ~3-5%

Model:
Hit rate: 15-25%
→ 3x do 5x bolje od random

To je stvarna vrijednost koja se može izmjeriti
i dokazati A/B testom nakon implementacije

Moja preporuka kako to prodati
1. Implementiraj pipeline
2. Pokreni na historijskim podacima
3. Uzmi period koji znaš ishod (npr. 2023)
   → Treniraj na podacima do 2022
   → Provjeri da li su preporuke za 2023 tačne
4. Izračunaj:
   - Precision@3
   - Usporedi sa random baseline
   - Pretvori u KM (potencijalni prihod)
5. Prezentuj: "Model je 4x bolji od nasumičnog pristupa
               i može generisati X KM dodatnog prihoda"
To je priča koja se prodaje – ne tehnički detalji o embeddinzima.